# Data Augmentation Pipeline — `chord_data_1217`

This notebook generates augmented versions of all audio files in `chord_data_1217` and produces the corresponding `.jams` annotation files for each augmentation.

**Augmentation types applied:**
- Convolutional reverb (IR-based)
- Real-world noise from FreeSound (café, street, wind)
- Random EQ
- Dynamic compression

**Output structure:**
```
final_augmentation/
├── data_augmentation_files/   ← augmented .mp3 audio files
└── data_augmentation_jams/    ← copied .jams (one per augmented audio)
```

---
## ⚙️ Step 0 — Install dependencies

In [1]:
!pip install git+https://github.com/MTG/freesound-python.git librosa soundfile scipy numpy jams

  Cloning https://github.com/MTG/freesound-python.git to C:\Users\User\AppData\Local\Temp\pip-req-build-e_f1c0ab
  Resolved https://github.com/MTG/freesound-python.git to commit 73cf6d14f7ce8174d943bdc78ff30f99878a5db8
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/MTG/freesound-python.git 'C:\Users\User\AppData\Local\Temp\pip-req-build-e_f1c0ab'


---
## 🔑 Step 1 — FreeSound API authentication

### How to get your API credentials:
1. Log in at [https://freesound.org](https://freesound.org) with your UPF account.
2. Go to **your username → Settings → API credentials** (or directly to [https://freesound.org/apiv2/apply/](https://freesound.org/apiv2/apply/)).
3. Create a new application — name it anything (e.g. `ACE_augmentation`).
4. Copy your **Client ID** and **Client Secret**.
5. For this notebook we use the **Client Credentials flow** (no user login needed), which gives you a token to download sounds with a Creative Commons license.

> ⚠️ **Never commit your credentials to git.** Use environment variables or a local `.env` file.

In [1]:
import os

# ── Paste your credentials here (or load from environment) ──────────────────
FREESOUND_API_KEY = os.environ.get("FREESOUND_API_KEY", "H7F3clGsFSWXhlEUKfiIJtkxCy9nYXfkdUfmIqEQ")
# ─────────────────────────────────────────────────────────────────────────────

import freesound
fs_client = freesound.FreesoundClient()
fs_client.set_token(FREESOUND_API_KEY, "token")
print("FreeSound client ready.")

FreeSound client ready.


c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


---
## 📁 Step 2 — Project paths

In [2]:
from pathlib import Path

# ── Adjust these paths to your local setup ───────────────────────────────────
CHORD_DATA_DIR   = Path("chord_data_1217")
AUDIO_DIR        = CHORD_DATA_DIR / "audio"
JAMS_DIR         = CHORD_DATA_DIR / "references_v2"
IR_DIR           = Path("irs")           # folder with your impulse responses
VOCAB_PATH       = Path("final_augmentation\chords_vocab.joblib") # path to chord vocabulary

OUTPUT_ROOT      = Path("final_augmentation")
OUTPUT_AUDIO     = OUTPUT_ROOT / "data_augmentation_files"
OUTPUT_JAMS      = OUTPUT_ROOT / "data_augmentation_jams"
NOISE_CACHE_DIR  = OUTPUT_ROOT / "noise_cache"  # downloaded FreeSound clips

for d in [OUTPUT_AUDIO, OUTPUT_JAMS, NOISE_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths OK.")
print(f"  Audio dir   : {AUDIO_DIR}")
print(f"  JAMS dir    : {JAMS_DIR}")
print(f"  Output audio: {OUTPUT_AUDIO}")
print(f"  Output JAMS : {OUTPUT_JAMS}")

Paths OK.
  Audio dir   : chord_data_1217\audio
  JAMS dir    : chord_data_1217\references_v2
  Output audio: final_augmentation\data_augmentation_files
  Output JAMS : final_augmentation\data_augmentation_jams


---
## 🔍 Step 3 — Discover audio files

In [3]:
audio_files = sorted(AUDIO_DIR.rglob("*.mp3"))
print(f"Found {len(audio_files)} audio files.")
for f in audio_files[:5]:
    print(" ", f.name)

Found 1217 audio files.
  TR6R91L11C8A40D710.mp3
  TRACGVT149E3B9BE3F.mp3
  TRACPPB149E33C10B9.mp3
  TRADINA127F847B84E.mp3
  TRAEQJQ149E3BA694B.mp3


---
## 🌐 Step 4 — Download background noises from FreeSound

We download a small pool of clips per category. Each clip is saved to `noise_cache/`.
Clips are concatenated/trimmed at augmentation time to match the target audio length.

### FreeSound search tags used:
| Category | Tags |
|---|---|
| Café | `café ambience`, `coffee shop noise` |
| Street / traffic | `city street`, `traffic noise` |
| Wind | `wind outdoor`, `wind noise` |

In [4]:
import requests
import time

NOISE_QUERIES = {
    "cafe":   ["cafe ambience", "coffee shop background noise"],
    "street": ["city street ambience", "traffic noise urban"],
    "wind":   ["wind outdoor ambience", "wind noise exterior"],
}

# Number of clips to download per category (adjust to taste)
CLIPS_PER_CATEGORY = 3


def download_freesound_clips(
    client: freesound.FreesoundClient,
    queries: dict[str, list[str]],
    output_dir: Path,
    clips_per_category: int = 3,
    min_duration: float = 10.0,   # seconds — skip very short clips
    max_duration: float = 300.0,  # seconds
) -> dict[str, list[Path]]:
    """Search FreeSound and download HQ-preview MP3s for each noise category.
    Uses client.search() which is the correct method in the freesound PyPI package.
    Returns a dict mapping category -> list of local paths.
    """
    category_files: dict[str, list[Path]] = {cat: [] for cat in queries}

    for category, query_list in queries.items():
        cat_dir = output_dir / category
        cat_dir.mkdir(parents=True, exist_ok=True)
        collected = 0

        for query in query_list:
            if collected >= clips_per_category:
                break
            print(f"[{category}] Searching: '{query}'...")
            try:
                # freesound PyPI package exposes client.search()
                results = client.search(
                    query=query,
                    filter=f"duration:[{min_duration} TO {max_duration}]",
                    fields="id,name,duration,previews,license",
                    page_size=10,
                )
            except Exception as e:
                print(f"  Search error: {e}")
                continue

            for sound in results:
                if collected >= clips_per_category:
                    break

                dest_mp3 = cat_dir / f"{category}_{sound.id}.mp3"
                if dest_mp3.exists():
                    print(f"  [cached] {dest_mp3.name}")
                    category_files[category].append(dest_mp3)
                    collected += 1
                    continue

                # HQ preview does not require OAuth — just the API key token
                try:
                    preview_url = sound.previews.preview_hq_mp3
                    r = requests.get(
                        preview_url,
                        headers={"Authorization": f"Token {FREESOUND_API_KEY}"},
                        timeout=30,
                    )
                    r.raise_for_status()
                    dest_mp3.write_bytes(r.content)
                    print(f"  [downloaded] {dest_mp3.name}  ({sound.duration:.1f}s)")
                    category_files[category].append(dest_mp3)
                    collected += 1
                    time.sleep(0.3)  # be polite to the API
                except Exception as e:
                    print(f"  Download error for sound {sound.id}: {e}")

    return category_files


noise_files = download_freesound_clips(
    fs_client, NOISE_QUERIES, NOISE_CACHE_DIR, clips_per_category=CLIPS_PER_CATEGORY
)

for cat, files in noise_files.items():
    print(f"{cat}: {len(files)} clip(s) available")

[cafe] Searching: 'cafe ambience'...
  [cached] cafe_366483.mp3
  [cached] cafe_363713.mp3
  [cached] cafe_516435.mp3
[street] Searching: 'city street ambience'...
  [cached] street_622736.mp3
  [cached] street_608157.mp3
  [cached] street_723607.mp3
[wind] Searching: 'wind outdoor ambience'...
  [cached] wind_138294.mp3
  [cached] wind_326908.mp3
  [cached] wind_617747.mp3
cafe: 3 clip(s) available
street: 3 clip(s) available
wind: 3 clip(s) available


---
## 🎛️ Step 5 — Augmentation configuration

All wet/dry ratios default to **0.5 (50 % wet / 50 % dry)**.

In [5]:
import re
import random
from pathlib import Path

SR = 22050  # sample rate used throughout
AUGMENTATION_SEED = 42

# Impulse responses available locally
ir_files = sorted(IR_DIR.rglob("*.wav")) + sorted(IR_DIR.rglob("*.flac"))
print(f"Found {len(ir_files)} impulse response file(s).")

# Build augmentation plan: list of (suffix, [(fn_name, args_dict), ...])
# Each entry produces one output file per source audio.
# Multiple (fn_name, args_dict) pairs are applied in sequence (chained).

AUGMENTATION_PLAN = []

# -- Reverb: one augmentation per IR file
for ir_path in ir_files:
    slug = re.sub(r"[^A-Za-z0-9]+", "-", ir_path.stem).strip("-").lower()
    AUGMENTATION_PLAN.append((
        f"reverb_{slug}",
        [("apply_reverb", {"ir_path": str(ir_path), "sr": SR, "wet_dry_mix": 0.5})],
    ))

# -- Real noise: add each noise category to every existing augmentation
_crng = random.Random(AUGMENTATION_SEED)
AUGMENTATION_PLAN_WITH_NOISE = []
randomeq_step = ("randomly_eq", {"sr": SR, "gain_range": (-6, 6)})
for category, files in noise_files.items():
    if files:
        noise_step = ("add_real_noise", {"noise_files": files, "sr": SR, "wet_dry_mix": 0.5})
        for suffix, steps in AUGMENTATION_PLAN:
            compression_step = ("apply_compression", {
                "sr":           SR,
                "threshold_db": _crng.uniform(-30, -10),   # dB: light to heavy gain reduction
                "ratio":        _crng.uniform(2.0, 8.0),   # 2:1 (gentle) to 8:1 (heavy)
                "attack_ms":    _crng.uniform(1.0, 30.0),  # ms: fast to moderate attack
                "release_ms":   _crng.uniform(50, 400),    # ms: snappy to slow release
            })
            AUGMENTATION_PLAN_WITH_NOISE.append((
                f"{suffix}_noise_{category}",
                steps + [noise_step] + [randomeq_step] + [compression_step],
            ))

AUGMENTATION_PLAN = AUGMENTATION_PLAN_WITH_NOISE

print(f"\nAugmentation plan: {len(AUGMENTATION_PLAN)} augmentation type(s)")
for suffix, steps in AUGMENTATION_PLAN:
    fns = " → ".join(fn for fn, _ in steps)
    print(f"  [{fns}]  suffix → _{suffix}")

Found 34 impulse response file(s).

Augmentation plan: 102 augmentation type(s)
  [apply_reverb → add_real_noise → randomly_eq → apply_compression]  suffix → _reverb_1st-baptist-nashville-balcony_noise_cafe
  [apply_reverb → add_real_noise → randomly_eq → apply_compression]  suffix → _reverb_1st-baptist-nashville-balcony_noise_cafe
  [apply_reverb → add_real_noise → randomly_eq → apply_compression]  suffix → _reverb_1st-baptist-nashville-far-close_noise_cafe
  [apply_reverb → add_real_noise → randomly_eq → apply_compression]  suffix → _reverb_1st-baptist-nashville-far-wide_noise_cafe
  [apply_reverb → add_real_noise → randomly_eq → apply_compression]  suffix → _reverb_crash-ir_noise_cafe
  [apply_reverb → add_real_noise → randomly_eq → apply_compression]  suffix → _reverb_hh-ir_noise_cafe
  [apply_reverb → add_real_noise → randomly_eq → apply_compression]  suffix → _reverb_kick-ir_noise_cafe
  [apply_reverb → add_real_noise → randomly_eq → apply_compression]  suffix → _reverb_ride-ir_n

---
## 🚀 Step 6 — Run augmentation pipeline

For each source audio file and each augmentation type:
1. Generate `<stem>_<suffix>.mp3` in `data_augmentation_files/`
2. Copy the corresponding `.jams` to `data_augmentation_jams/<stem>_<suffix>.jams`

In [6]:
import shutil
import random
import traceback
import json
import librosa
import soundfile as sf
from tqdm.auto import tqdm
from data_augmentation_merged import (
    apply_reverb_arr,
    add_real_noise_arr,
    randomly_eq_arr,
    apply_compression_arr,
)

AUGMENTATION_FN_MAP = {
    "apply_reverb":      apply_reverb_arr,
    "add_real_noise":    add_real_noise_arr,
    "randomly_eq":       randomly_eq_arr,
    "apply_compression": apply_compression_arr,
}

AUGMENTATIONS_PER_FILE = 4
RANDOM_SEED = 42
CORPUS_FILTER = "Billboard-Chords"  # set to None to disable filtering


def get_jams_path(audio_path: Path, jams_dir: Path) -> Path | None:
    """Return the .jams file that corresponds to an audio file, or None."""
    candidate = jams_dir / (audio_path.stem + ".jams")
    return candidate if candidate.exists() else None


def jams_has_corpus(jams_path: Path, corpus: str) -> bool:
    """Return True if any annotation in the JAMS file matches the given corpus."""
    with open(jams_path) as f:
        data = json.load(f)
    return any(
        ann.get("annotation_metadata", {}).get("corpus", "") == corpus
        for ann in data.get("annotations", [])
    )


def resolve_args(args: dict) -> dict:
    """For add_real_noise: pick one random clip from the pool at runtime."""
    resolved = dict(args)
    if "noise_files" in resolved:
        resolved["noise_path"] = str(random.choice(resolved.pop("noise_files")))
    return resolved


ok_count = 0
skip_count = 0
error_count = 0

rng = random.Random(RANDOM_SEED)

for audio_path in tqdm(audio_files, desc="Augmenting", unit="file"):
    jams_src = get_jams_path(audio_path, JAMS_DIR)
    if jams_src is None:
        print(f"[WARN] No .jams found for {audio_path.name} — skipping.")
        skip_count += 1
        continue

    if CORPUS_FILTER is not None and not jams_has_corpus(jams_src, CORPUS_FILTER):
        skip_count += 1
        continue

    # Copy original audio + jams to output folders alongside augmented versions
    if not (OUTPUT_AUDIO / audio_path.name).exists():
        shutil.copy2(audio_path, OUTPUT_AUDIO / audio_path.name)
    if not (OUTPUT_JAMS / jams_src.name).exists():
        shutil.copy2(jams_src, OUTPUT_JAMS / jams_src.name)

    selected = rng.sample(AUGMENTATION_PLAN, k=AUGMENTATIONS_PER_FILE)

    for suffix, steps in selected:
        out_stem  = f"{audio_path.stem}_{suffix}"
        out_audio = OUTPUT_AUDIO / f"{out_stem}.mp3"
        out_jams  = OUTPUT_JAMS  / f"{out_stem}.jams"

        if out_audio.exists() and out_jams.exists():
            skip_count += 1
            continue

        try:
            # Load once, chain steps fully in memory, write once
            y, sr = librosa.load(str(audio_path), sr=SR)

            for fn_name, base_args in steps:
                fn   = AUGMENTATION_FN_MAP[fn_name]
                args = resolve_args(base_args)
                y    = fn(y, sr, args)

            sf.write(str(out_audio), y, sr, format="MP3")
            shutil.copy2(jams_src, out_jams)
            ok_count += 1

        except Exception as e:
            print(f"[ERROR] {out_stem}: {e}")
            traceback.print_exc()
            error_count += 1

print(f"\n=== Pipeline summary ===")
print(f"  Generated : {ok_count}")
print(f"  Skipped   : {skip_count}")
print(f"  Errors    : {error_count}")

Augmenting:   0%|          | 0/1217 [00:00<?, ?file/s]


=== Pipeline summary ===
  Generated : 2927
  Skipped   : 504
  Errors    : 0


---
## ✅ Step 7 — Verification

Check that every audio file in the output has a matching `.jams`.

In [7]:
aug_audio_files = sorted(OUTPUT_AUDIO.glob("*.mp3"))
aug_jams_files  = sorted(OUTPUT_JAMS.glob("*.jams"))

audio_stems = {f.stem for f in aug_audio_files}
jams_stems  = {f.stem for f in aug_jams_files}

missing_jams  = audio_stems - jams_stems
orphan_jams   = jams_stems  - audio_stems

print(f"Augmented audio files : {len(aug_audio_files)}")
print(f"Augmented JAMS files  : {len(aug_jams_files)}")

if missing_jams:
    print(f"\n⚠️  Audio files WITHOUT matching .jams ({len(missing_jams)}):")
    for s in sorted(missing_jams): print(f"  {s}")
else:
    print("\n✅ Every audio file has a matching .jams.")

if orphan_jams:
    print(f"\n⚠️  Orphan .jams without audio ({len(orphan_jams)}):")
    for s in sorted(orphan_jams): print(f"  {s}")

Augmented audio files : 3681
Augmented JAMS files  : 3681

✅ Every audio file has a matching .jams.


## Preprocessing

## Training

In [1]:
import torch

def check_blackwell():
    print(f"--- Diagnóstico de Hardware ---")
    if torch.cuda.is_available():
        print(f"✅ ¡Conectado a la GPU!")
        print(f"Tarjeta: {torch.cuda.get_device_name(0)}")
        print(f"Capacidad de Cómputo: {torch.cuda.get_device_capability(0)}")
        
        # Test rápido: Multiplicación de matrices en GPU
        x = torch.randn(1000, 1000).cuda()
        y = torch.randn(1000, 1000).cuda()
        z = torch.matmul(x, y)
        print(f"🚀 Test de computación completado en la GPU.")
    else:
        print("❌ Sigue detectando CPU. Revisa que el entorno virtual esté activo.")

if __name__ == "__main__":
    check_blackwell()

--- Diagnóstico de Hardware ---
✅ ¡Conectado a la GPU!
Tarjeta: NVIDIA GeForce RTX 5060 Laptop GPU
Capacidad de Cómputo: (12, 0)
🚀 Test de computación completado en la GPU.


In [2]:
# Login to Weights & Biases for experiment tracking
import wandb
wandb.login()
import os
#os.environ['WANDB_API_KEY'] = 'api'

c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\User\_netrc.
wandb: Currently logged in as: rafaeleduardo-moncayo01 (rafaeleduardo-moncayo01-universitat-pompeu-fabra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
from pathlib import Path

In [6]:
from ace_safe_trainer import main as ace_train_model

train_data_path = "chord_data_1217\\billboard_for_training\\preprocesed_billboard_data"
vocab_path = "chord_data_1217\\chords_vocab.joblib"
checkpoint_path = "chord_data_1217\\billboard_for_training\\Checkpoints"
#checkpoint_decomposed_path = "chord_data_1217\\Decomposed_Checkpoints"

print(f"Base data path: {train_data_path}")

Base data path: chord_data_1217\billboard_for_training\preprocesed_billboard_data


In [7]:
ace_train_model(
    model_name="conformer",
    run_name="MIR_Augmentation_Conformer",
    params={
        "train.data_path": train_data_path,
        "train.vocab_path": vocab_path,
        "train.max_epochs": 10,
        "train.accelerator": "cuda",
        
        "ChocoAudioDataModule.num_workers": 1,
        
        "ModelCheckpoint.dirpath": checkpoint_path,
        "ModelCheckpoint.save_last": True 
    }
)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


Found 31917 training files and 0 test files.
Training data path: chord_data_1217\billboard_for_training\preprocesed_billboard_data
Test data path: chord_data_1217\billboard_for_training\preprocesed_billboard_data
Training IDs: ['Isophonics_TRCMANT149E2CA17A4', 'Isophonics_TRKXMSW149E3D25BEB', 'Isophonics_TRORMXT149E3670858', 'Isophonics_TROSSUK149E3AE03BD', 'Isophonics_TRVACGQ149E2CA49F8']... 
Test IDs: []... 
Train data path: chord_data_1217\billboard_for_training\preprocesed_billboard_data
Test data path: chord_data_1217\billboard_for_training\preprocesed_billboard_data
Leakage-safe split: 627 train sources, 111 val sources, 0 test sources


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                 | Type               | Params | Mode 
--------------------------------------------------------------------
0 | positional_encodings | PositionalEncoding | 0      | train
1 | input_projection     | Linear             | 37.1 K | train
2 | conformer            | Conformer          | 6.1 M  | train
3 | output_projection    | Linear             | 6.7 K  | train
4 | train_accuracy       | MulticlassAccuracy | 0      | train
5 | val_accuracy         | MulticlassAccuracy | 0      | train
6 | test_accuracy        | MulticlassAccuracy | 0      | train
--------------------------------------------------------------------
6.1 M     Trainable params
0         Non-trainable params
6.1 M     Total params
24.542    Total estimated model params size (MB)
136       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Leakage-safe split: 627 train sources, 111 val sources, 0 test sources


c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\lightning\pytorch\utilities\data.py:106: Total length of `DataLoader` across ranks is zero. Please make sure this was your intention.


epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
lr-AdamW,███▇▆▆▅▄▂▁
train_accuracy,▁▄▅▆▆▇▇▇██
train_loss,█▅▄▃▃▂▂▁▁▁
trainer/global_step,▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇██
val_accuracy,▄▆▁▇▅▅█▇█▇
val_loss,▂▁▄▃▄▄▄▆▆█
epoch,9
lr-AdamW,7e-05
train_accuracy,0.9154
train_loss,0.26141


In [8]:
train_data_path = "final_augmentation\\preprocessed_augmented_data"
vocab_path = "final_augmentation\\chords_vocab.joblib"
checkpoint_path = "final_augmentation\\Checkpoints"

print(f"Base data path: {train_data_path}")

Base data path: final_augmentation\preprocessed_augmented_data


In [9]:
ace_train_model(
    model_name="conformer",
    run_name="MIR_Augmentation_Conformer_With_Data_Augmentation",
    params={
        "train.data_path": train_data_path,
        "train.vocab_path": vocab_path,
        "train.max_epochs": 10,
        "train.accelerator": "cuda",
        
        "ChocoAudioDataModule.num_workers": 1,
        
        "ModelCheckpoint.dirpath": checkpoint_path,
        "ModelCheckpoint.save_last": True 
    }
)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Found 159237 training files and 0 test files.
Training data path: final_augmentation\preprocessed_augmented_data
Test data path: final_augmentation\preprocessed_augmented_data
Training IDs: ['Isophonics_TRCMANT149E2CA17A4_reverb_1st-baptist-nashville-far-close_noise_cafe', 'Isophonics_TRCMANT149E2CA17A4_reverb_1st-baptist-nashville-far-close_noise_wind', 'Isophonics_TRCMANT149E2CA17A4_reverb_ride-ir_noise_wind', 'Isophonics_TRCMANT149E2CA17A4_reverb_toma-ir_noise_street', 'Isophonics_TRCMANT149E2CA17A4']... 
Test IDs: []... 
Train data path: final_augmentation\preprocessed_augmented_data
Test data path: final_augmentation\preprocessed_augmented_data
Leakage-safe split: 3128 train sources, 553 val sources, 0 test sources


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                 | Type               | Params | Mode 
--------------------------------------------------------------------
0 | positional_encodings | PositionalEncoding | 0      | train
1 | input_projection     | Linear             | 37.1 K | train
2 | conformer            | Conformer          | 6.1 M  | train
3 | output_projection    | Linear             | 6.7 K  | train
4 | train_accuracy       | MulticlassAccuracy | 0      | train
5 | val_accuracy         | MulticlassAccuracy | 0      | train
6 | test_accuracy        | MulticlassAccuracy | 0      | train
--------------------------------------------------------------------
6.1 M     Trainable params
0         Non-trainable params
6.1 M     Total params
24.542    Total estimated model params size (MB)
136       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Leakage-safe split: 3128 train sources, 553 val sources, 0 test sources


epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
lr-AdamW,███▇▆▆▅▄▂▁
train_accuracy,▁▃▅▅▆▇▇▇██
train_loss,█▅▄▄▃▂▂▂▁▁
trainer/global_step,▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇██
val_accuracy,▁▃▄▅▆▆▇▇██
val_loss,█▆▅▄▃▂▂▂▁▁
epoch,9
lr-AdamW,7e-05
train_accuracy,0.88048
train_loss,0.35785


In [19]:
from pathlib import Path
from ACE.inference import run_inference

run_inference(
    audio_path="chord_data_1217\\audio\\TRAWVNL127FA2C9CD6.mp3",
    checkpoint="chord_data_1217\\billboard_for_training\\Checkpoints\\last.ckpt",
    model_name="conformer",
    vocab_path="chord_data_1217\\chords_vocab.joblib",
    out_lab=Path("chord_data_1217\\out_lab\\out_lab_inferenceNoAug.lab"),
    chord_min_duration=0.1
)

✅ Loaded model from chord_data_1217\billboard_for_training\Checkpoints\last.ckpt and vocab from chord_data_1217\chords_vocab.joblib
🔍 Processing 16 chunks of ~20.0s each
Chunk 1/16 (start 0.0s)
Chunk 2/16 (start 20.0s)
Chunk 3/16 (start 40.0s)
Chunk 4/16 (start 60.0s)
Chunk 5/16 (start 80.0s)
Chunk 6/16 (start 100.0s)
Chunk 7/16 (start 120.0s)
Chunk 8/16 (start 140.0s)
Chunk 9/16 (start 160.0s)
Chunk 10/16 (start 180.0s)
Chunk 11/16 (start 200.0s)
Chunk 12/16 (start 220.0s)
Chunk 13/16 (start 240.0s)
Chunk 14/16 (start 260.0s)
Chunk 15/16 (start 280.0s)
Chunk 16/16 (start 300.0s)
💾 Saved chord_data_1217\out_lab\out_lab_inferenceNoAug.lab


In [18]:
run_inference(
    audio_path="chord_data_1217\\audio\\TRAWVNL127FA2C9CD6.mp3",
    checkpoint="final_augmentation\\Checkpoints\\last.ckpt",
    vocab_path="final_augmentation\\chords_vocab.joblib",
    model_name="conformer",
    out_lab=Path("final_augmentation\\out_lab\\out_lab_WithAug.lab"),
    chord_min_duration=0.1
)

✅ Loaded model from final_augmentation\Checkpoints\last.ckpt and vocab from final_augmentation\chords_vocab.joblib
🔍 Processing 16 chunks of ~20.0s each
Chunk 1/16 (start 0.0s)
Chunk 2/16 (start 20.0s)
Chunk 3/16 (start 40.0s)
Chunk 4/16 (start 60.0s)
Chunk 5/16 (start 80.0s)
Chunk 6/16 (start 100.0s)
Chunk 7/16 (start 120.0s)
Chunk 8/16 (start 140.0s)
Chunk 9/16 (start 160.0s)
Chunk 10/16 (start 180.0s)
Chunk 11/16 (start 200.0s)
Chunk 12/16 (start 220.0s)
Chunk 13/16 (start 240.0s)
Chunk 14/16 (start 260.0s)
Chunk 15/16 (start 280.0s)
Chunk 16/16 (start 300.0s)
💾 Saved final_augmentation\out_lab\out_lab_WithAug.lab


## Testing
Using mir_eval jams

In [20]:
!pip install mir_eval jams

In [36]:
import torch
import lightning as L
from pathlib import Path
from ACE.models.conformer import ConformerModel
import gin
import numpy as np
import mir_eval
import joblib
import sys
import pandas as pd


In [22]:
# ── Ajusta estas rutas ────────────────────────────────────────────────────────
VOCAB_PATH       = Path("chord_data_1217\\chords_vocab.joblib")
CHECKPOINT_BASE  = Path("chord_data_1217\\billboard_for_training\\Checkpoints\\last.ckpt")
CHECKPOINT_AUG   = Path("final_augmentation\\Checkpoints\\last.ckpt")
DATA_PATH        = Path("chord_data_1217\preprocessed_data_og")   # carpeta con los .pt de MARL
# ─────────────────────────────────────────────────────────────────────────────

gin.clear_config()
gin.parse_config_file("ACE/trainer.gin")

vocabularies = {
    "simplified": 50, "root": 13, "bass": 13, "mode": 8,
    "majmin": 26, "onehot": 12, "complete": 170,
}

def load_model(checkpoint_path: Path) -> ConformerModel:
    model = ConformerModel.load_from_checkpoint(
        checkpoint_path,
        vocabularies=vocabularies,
        vocab_path=str(VOCAB_PATH),
        map_location="cpu",
    )
    model.eval()
    return model

model_base = load_model(CHECKPOINT_BASE)
model_aug  = load_model(CHECKPOINT_AUG)
print("Both models loaded OK.")

Both models loaded OK.


In [ ]:
sys.path.insert(0, ".")  
from ace_safe_trainer import ChocoAudioDataModule

datamodule = ChocoAudioDataModule(
    data_path=DATA_PATH,
    batch_size=64,
    num_workers=1,
    augmentation=False, 
)
datamodule.prepare_data()
datamodule.setup(stage="test")

test_loader = datamodule.test_dataloader()
print(f"Test batches: {len(test_loader)}")

Found 6568 training files and 14723 test files.
Training data path: chord_data_1217\preprocessed_data_og
Test data path: chord_data_1217\preprocessed_data_og
Training IDs: ['Isophonics_TRAHXQW149E3BCE2C1', 'Isophonics_TRAITGI149E3C71235', 'Isophonics_TRAKIXJ149E332D53F', 'Isophonics_TRATLJQ149E33D7D2A', 'Isophonics_TRAXWBI149E3D6BC67']... 
Test IDs: ['MARL_TR6R91L11C8A40D710', 'MARL_TRADINA127F847B84E', 'MARL_TRAHMSN127F92CD4AD', 'MARL_TRALJVL127F98F7094', 'MARL_TRAPEUF149E3BF4C4C']... 
Leakage-safe split: 163 train sources, 29 val sources, 287 test sources
Test batches: 60


In [33]:
vocab_le = joblib.load(VOCAB_PATH)

# Parámetros del preprocesado CQT — ajusta si son distintos en tu pipeline
HOP_LENGTH  = 4096
SAMPLE_RATE = 22050
FRAME_DUR   = HOP_LENGTH / SAMPLE_RATE  # segundos por frame ≈ 0.1856 s

def idx_to_chord_list(idx_array: np.ndarray) -> list[str]:
    """Convert index array to chord strings via LabelEncoder."""
    return vocab_le.inverse_transform(idx_array.astype(int)).tolist()

def make_intervals(n_frames: int) -> np.ndarray:
    """Build (n_frames, 2) interval array with no floating point overlap."""
    indices = np.arange(n_frames, dtype=np.float64)
    starts  = indices * FRAME_DUR
    ends    = (indices + 1.0) * FRAME_DUR
    # Garantiza que end[i] == start[i+1] exactamente
    ends[:-1] = starts[1:]
    return np.stack([starts, ends], axis=1)

def run_inference(model: torch.nn.Module, loader) -> dict:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = model.to(device)
    model.eval()

    scores_per_song = []

    with torch.no_grad():
        for batch in loader:
            cqt        = batch[0].to(device)      # (B, 1, 144, 108)
            ref_idx    = batch[1]["majmin"].cpu()  # (B, 108)

            logits  = model(cqt)                   # (B, 108, 26)
            est_idx = logits.argmax(dim=-1).cpu()  # (B, 108)

            n_frames  = cqt.shape[-1]              # 108
            intervals = make_intervals(n_frames)   # (108, 2)

            for i in range(cqt.size(0)):
                ref_chords = idx_to_chord_list(ref_idx[i].numpy())
                est_chords = idx_to_chord_list(est_idx[i].numpy())

                try:
                    score = mir_eval.chord.evaluate(
                        intervals, ref_chords,
                        intervals, est_chords,
                    )
                    scores_per_song.append(score)
                except Exception as e:
                    print(f"  [mir_eval error] sample {i}: {e}")
                    continue

    if not scores_per_song:
        raise RuntimeError("No songs scored — check shapes or vocab.")

    agg = {}
    for key in scores_per_song[0].keys():
        agg[key] = float(np.mean([s[key] for s in scores_per_song]))

    return {"per_song": scores_per_song, "aggregate": agg}


results_base = run_inference(model_base, test_loader)
results_aug  = run_inference(model_aug,  test_loader)
print("Inference complete.")

Inference complete.


In [ ]:
metrics_of_interest = [
    "thirds",       # majmin simplificado
    "triads",       # tríadas
    "tetrads",      # tétradas (7th chords)
    "majmin",       # major/minor
    "mirex",        # métrica oficial MIREX
    "root",         # solo root
    "bass",         # bajo
]

rows = []
for metric in metrics_of_interest:
    base_val = results_base["aggregate"].get(metric, float("nan"))
    aug_val  = results_aug["aggregate"].get(metric, float("nan"))
    delta    = aug_val - base_val
    rows.append({
        "Metric":        metric,
        "Baseline":      round(base_val, 4),
        "Aug model":     round(aug_val,  4),
        "Δ (aug−base)":  round(delta,    4),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

 Metric  Baseline  Aug model  Δ (aug−base)
 thirds    0.7848     0.7951        0.0103
 triads    0.6999     0.7140        0.0141
tetrads    0.6704     0.6836        0.0132
 majmin    0.6566     0.6664        0.0099
  mirex    0.7809     0.7803       -0.0006
   root    0.8387     0.8495        0.0108
   bass       NaN        NaN           NaN


In [ ]:
from scipy import stats

for key in ["thirds", "triads", "tetrads", "majmin", "mirex", "root"]:
    base_scores = [s[key] for s in results_base["per_song"] if not np.isnan(s[key])]
    aug_scores  = [s[key] for s in results_aug["per_song"]  if not np.isnan(s[key])]
    
    # Wilcoxon signed-rank test
    stat, p = stats.wilcoxon(base_scores, aug_scores)
    sig = "✅ significativo" if p < 0.05 else "⚠️ no significativo"
    print(f"{key:10s}  p={p:.4f}  {sig}")

thirds      p=0.0000  ✅ significativo
triads      p=0.0000  ✅ significativo
tetrads     p=0.0000  ✅ significativo
majmin      p=0.0002  ✅ significativo
mirex       p=0.1702  ⚠️ no significativo
root        p=0.0000  ✅ significativo
